# QPINN: screened Poisson

Domain: $\Omega=(-1,1)^2$ with homogeneous Dirichlet boundary condition $u=0$ on $\partial\Omega$.

PDE:

$$
-\Delta u(x)+\lambda u(x)=q(x),
\qquad x=(x_1,x_2)\in\Omega.
$$

With $\phi_m(t)=\cos(m\pi t/2)$,

$$
q(x_1,x_2)=\phi_1(x_1)\phi_1(x_2)
+0.3\left[\phi_1(x_1)\phi_3(x_2)+\phi_3(x_1)\phi_1(x_2)\right].
$$

Analytical solution:

$$
u_\star(x_1,x_2)=
\frac{\phi_1(x_1)\phi_1(x_2)}{\lambda+\pi^2/2}
+\frac{0.3\left[\phi_1(x_1)\phi_3(x_2)+\phi_3(x_1)\phi_1(x_2)\right]}{\lambda+5\pi^2/2}.
$$

Models: Ordinary QFM, Cheb-QFM, exact $B_2$-QFM, randomized $B_2$-QFM, and a fully connected PINN baseline. The shared circuit, PDE loss, training, and plotting code lives in the `qmps` package.


## Parameters

Set the PDE, circuit, FC PINN, training, output-directory, and figure-resolution parameters.


In [1]:
from __future__ import annotations

import qmps as qmp
import qmps.experiments as qexp
from qmps.problems import ScreenedPoissonProblem

# Runtime.
USE_DOUBLE_PRECISION = False
SEED = 5

# Problem and circuit size.
DIM = 2
N_QUBITS = 2
N_UPLOAD_LAYERS = 2
STRONG_LAYERS_PER_BLOCK = 2

# Circuit encoding and output scaling.
EXP_BASE = 3.0
ENCODING_SCALE = 1.0
OUTPUT_SCALE = 1.0
DIFF_GENERATOR_PER_LAYER = False
ACOS_EPS = 1.0e-6

# Fully connected PINN baseline.
FC_HIDDEN_LAYERS = (4, 4)

# Screened Poisson PDE.
LAMBDA = 1.0
SOURCE_MODE_COEFFICIENTS = {
    (1, 1): 1.0,
    (1, 3): 0.3,
    (3, 1): 0.3,
}

# Randomized B2 average.
RANDOMIZED_B2_SAMPLES = 6

# Training.
TRAINABLE = False
TRAINING_STEPS = 300
N_RUNS = 20
INTERIOR_BATCH = 50
BOUNDARY_BATCH = 50
LR = 1.0e-1
LR_DECAY = 0.99
LR_MIN = 1.0e-4
INTERIOR_MARGIN = 0.96
GRAD_CLIP = 5.0
SOFT_BOUNDARY_WEIGHT = 10.0

# Evaluation and plotting.
EVAL_GRID_N = 50
FIGURE_DIR = "figures"
TRAINING_DATA_DIR = "training_data/poisson"
FIGURE_DPI = 300  # Use 600 for high-resolution export.
LOSS_YLIM = (1.0e-5, 1.0e2)
LOSS_YTICKS = [1.0e-5, 1.0e-4, 1.0e-3, 1.0e-2, 1.0e-1, 1.0, 1.0e1, 1.0e2]
FIGURE_PREFIX = "qpinn_screened_poisson"

runtime = qexp.setup_runtime(
    USE_DOUBLE_PRECISION,
    SEED,
    figure_dir=FIGURE_DIR,
    data_dir=TRAINING_DATA_DIR,
)
problem = ScreenedPoissonProblem(
    lambda_=LAMBDA,
    source_mode_coefficients=SOURCE_MODE_COEFFICIENTS,
)
qfm_config = qmp.QFMConfig(
    dim=DIM,
    n_qubits=N_QUBITS,
    n_upload_layers=N_UPLOAD_LAYERS,
    strong_layers_per_block=STRONG_LAYERS_PER_BLOCK,
    exp_base=EXP_BASE,
    encoding_scale=ENCODING_SCALE,
    output_scale=OUTPUT_SCALE,
    diff_generator_per_layer=DIFF_GENERATOR_PER_LAYER,
    acos_eps=ACOS_EPS,
    randomized_b2_samples=RANDOMIZED_B2_SAMPLES,
)
fc_config = qexp.FullyConnectedPINNConfig(
    dim=DIM,
    hidden_layers=FC_HIDDEN_LAYERS,
)
training_config = qexp.TrainingConfig(
    steps=TRAINING_STEPS,
    n_runs=N_RUNS,
    interior_batch=INTERIOR_BATCH,
    boundary_batch=BOUNDARY_BATCH,
    lr=LR,
    lr_decay=LR_DECAY,
    lr_min=LR_MIN,
    interior_margin=INTERIOR_MARGIN,
    grad_clip=GRAD_CLIP,
)
MODEL_SPECS = {
    "Ordinary QFM": {"group": "none", "input_map": "raw"},
    "Cheb-QFM": {"group": "none", "input_map": "chebyshev"},
    "$B_2$-QFM": {"group": "hyperoctahedral", "input_map": "raw"},
    "Randomized $B_2$-QFM": {
        "group": "randomized_hyperoctahedral",
        "input_map": "raw",
        "group_sample_count": RANDOMIZED_B2_SAMPLES,
    },
}


Using device=mps, real dtype=torch.float32, state representation=real/imag pair


## Models And Losses

Build the four quantum model variants, the FC PINN baseline, and the hard/soft loss functions.


In [2]:
def make_models(seed: int = SEED, hard_boundary: bool = True):
    return qmp.make_qfm_models(
        qfm_config,
        MODEL_SPECS,
        seed=seed,
        hard_boundary=hard_boundary,
        dtype=runtime.dtype,
        device=runtime.device,
    )


def make_fc_models(seed: int = SEED, hard_boundary: bool = True):
    return qexp.make_fully_connected_pinn_models(
        fc_config,
        seed=seed,
        hard_boundary=hard_boundary,
        dtype=runtime.dtype,
        device=runtime.device,
    )


def hard_loss(model, interior_x, boundary_x):
    return problem.loss(model, interior_x, boundary_x, boundary_weight=0.0)


def soft_loss(model, interior_x, boundary_x):
    return problem.loss(model, interior_x, boundary_x, boundary_weight=SOFT_BOUNDARY_WEIGHT)


loss_functions = qexp.make_loss_functions(hard_loss, soft_loss)
models = make_models(seed=SEED, hard_boundary=True)
fc_models = make_fc_models(seed=SEED, hard_boundary=True)
qexp.print_parameter_counts({**models, **fc_models})


Trainable parameter counts:
  Ordinary QFM            : 36
  Cheb-QFM                : 36
  $B_2$-QFM               : 36
  Randomized $B_2$-QFM    : 36
  Fully connected PINN    : 37


## Training Data

Train only when `TRAINABLE = True`; otherwise load the saved local single-run training-data files for plotting.


In [3]:
if TRAINABLE:
    result = qexp.train_and_save_boundary_comparison_runs(
        make_models=make_models,
        make_fc_models=make_fc_models,
        loss_functions=loss_functions,
        model_names=list(MODEL_SPECS),
        config=training_config,
        runtime=runtime,
        dim=DIM,
        seed=SEED,
        problem=problem,
        grid_n=EVAL_GRID_N,
        figure_prefix=FIGURE_PREFIX,
    )
else:
    result = qexp.load_training_data(
        runtime.data_dir,
        FIGURE_PREFIX,
        make_models=make_models,
        make_fc_models=make_fc_models,
    )
    training_data_paths = qexp.available_training_data_files(runtime.data_dir, FIGURE_PREFIX)
    for training_data_path in training_data_paths:
        print(f"loaded training data: {training_data_path}")


loaded training data: training_data/poisson/qpinn_screened_poisson_run_001_training_data.pt
loaded training data: training_data/poisson/qpinn_screened_poisson_run_002_training_data.pt
loaded training data: training_data/poisson/qpinn_screened_poisson_run_003_training_data.pt
loaded training data: training_data/poisson/qpinn_screened_poisson_run_004_training_data.pt
loaded training data: training_data/poisson/qpinn_screened_poisson_run_005_training_data.pt
loaded training data: training_data/poisson/qpinn_screened_poisson_run_006_training_data.pt
loaded training data: training_data/poisson/qpinn_screened_poisson_run_007_training_data.pt
loaded training data: training_data/poisson/qpinn_screened_poisson_run_008_training_data.pt
loaded training data: training_data/poisson/qpinn_screened_poisson_run_009_training_data.pt
loaded training data: training_data/poisson/qpinn_screened_poisson_run_010_training_data.pt


## Figures

Save the side-by-side loss figure and the combined solution-comparison figure. The solution panels compare the reference solution with the representative trained model for each constraint/model; the representative run is selected by final training residual for hard constraint and final training objective for soft constraint. With `N_RUNS = 1`, every panel uses run 1.


In [4]:
figure_paths = [
    qexp.plot_final_loss_comparison(
        result,
        runtime.figure_dir,
        FIGURE_PREFIX,
        dpi=FIGURE_DPI,
        loss_ylim=LOSS_YLIM,
        loss_yticks=LOSS_YTICKS,
    ),
    qexp.plot_final_solution_comparison(
        result,
        problem,
        runtime,
        dim=DIM,
        grid_n=EVAL_GRID_N,
        figure_prefix=FIGURE_PREFIX,
        dpi=FIGURE_DPI,
    ),
]
for figure_path in figure_paths:
    print(f"saved figure: {figure_path}")

qexp.print_final_metric_summary(result)


saved figure: figures/qpinn_screened_poisson_loss_comparison.png
saved figure: figures/qpinn_screened_poisson_solution_comparison.png

Final metrics (mean +/- std over runs)

Hard constraint
model                                       final loss         relative L2 error
QFM                            7.206e-03 +/- 4.518e-03   3.064e-02 +/- 1.758e-02
QCM                            8.160e-03 +/- 6.332e-03   3.186e-02 +/- 1.605e-02
$B_2$-QFM                      4.451e-04 +/- 8.998e-04   2.594e-03 +/- 2.370e-03
Randomized $B_2$-QFM           2.974e-03 +/- 2.638e-03   1.986e-02 +/- 1.346e-02
Fully connected PINN           7.440e-03 +/- 5.128e-03   2.381e-02 +/- 1.504e-02

Soft constraint
model                                       final loss         relative L2 error
QFM                            9.520e-02 +/- 6.185e-02   4.200e-01 +/- 7.099e-02
QCM                            3.424e-01 +/- 3.965e-01   8.171e-01 +/- 6.817e-01
$B_2$-QFM                      5.536e-02 +/- 2.014e-02   3.907e